--Grocery Inventory Adjusted Fix

-- This adds Date_Received, Product_ID, and Reorder_Level to grocery_inventory_adjusted--

In [ ]:
%sql
CREATE OR REPLACE TABLE grocery_inventory_adjusted AS
SELECT
  gi.Product_Name,
  gi.Category,
  gi.Warehouse_Location,
  DATE_ADD(gi.Expiration_Date, 649) AS Expiration_Date,
  DATEDIFF(DATE_ADD(gi.Expiration_Date, 649), CURRENT_DATE()) AS days_until_expiration,
  gi.Stock_Quantity,
  gi.Unit_Price,
  gi.Sales_Volume,
  gi.Inventory_Turnover_Rate,
  DATE_ADD(gi.Last_Order_Date, 649) AS Last_Order_Date,
  gi.Status,
  gi.Supplier_Name,
  gi.Product_ID,
  gi.Reorder_Level,
  DATE_ADD(gi.Date_Received, 649) AS Date_Received,
  (gi.Stock_Quantity * gi.Unit_Price) AS inventory_value_at_risk
FROM grocery_inventory gi
ORDER BY Expiration_Date;



SyntaxError: invalid syntax (ipython-input-2819066441.py, line 2)

-- KPI 1: Total Inventory Value

In [ ]:
%sql
-- KPI 1: Total Inventory Value
SELECT
  SUM(Stock_Quantity * Unit_Price) AS total_inventory_value
FROM grocery_inventory_adjusted;

total_inventory_value
332654.70999999996


In [ ]:
%sql
-- KPI 2: Number of Low-Stock Products
SELECT
  COUNT(*) AS low_stock_count
FROM grocery_inventory_adjusted
WHERE Stock_Quantity < Reorder_Level;

low_stock_count
455


In [ ]:
%sql
-- KPI 3: Average Turnover Rate
SELECT
  AVG(Inventory_Turnover_Rate) AS avg_turnover_rate
FROM grocery_inventory_adjusted;

avg_turnover_rate
50.15050505050505


-- VISUAL 2: BAR CHART - Top 10 Products by Sales
--

In [ ]:
%sql
SELECT
  Product_Name,
  Sales_Volume
FROM grocery_inventory_adjusted
ORDER BY Sales_Volume DESC
LIMIT 10;

Product_Name,Sales_Volume
Corn Oil,100
Haddock,100
Cheddar Cheese,100
Egg (Chicken),100
Jasmine Rice,100
Parmesan Cheese,100
Arabica Coffee,100
Lime,100
Vegetable Oil,100
Egg (Quail),100


-- VISUAL 3: LINE CHART - Monthly Inventory Value Trend
--

In [ ]:
%sql
SELECT
  DATE_TRUNC('month', Date_Received) AS month,
  SUM(Stock_Quantity * Unit_Price) AS total_inventory_value
FROM grocery_inventory_adjusted
GROUP BY DATE_TRUNC('month', Date_Received)
ORDER BY month;


month,total_inventory_value
2025-12-01T00:00:00.000Z,20790.25
2026-01-01T00:00:00.000Z,31902.96
2026-02-01T00:00:00.000Z,19256.75
2026-03-01T00:00:00.000Z,24790.82
2026-04-01T00:00:00.000Z,30966.350000000002
2026-05-01T00:00:00.000Z,30135.369999999995
2026-06-01T00:00:00.000Z,33295.13
2026-07-01T00:00:00.000Z,21702.05
2026-08-01T00:00:00.000Z,30851.64
2026-09-01T00:00:00.000Z,24203.3


- VISUAL 4: TABLE - Low Stock Alerts

In [ ]:
%sql
SELECT
  Product_Name,
  Category,
  Stock_Quantity,
  Reorder_Level,
  (Reorder_Level - Stock_Quantity) AS shortage_amount
FROM grocery_inventory_adjusted
WHERE Stock_Quantity < Reorder_Level
ORDER BY shortage_amount DESC;

Product_Name,Category,Stock_Quantity,Reorder_Level,shortage_amount
Heavy Cream,Dairy,14,99,85
Cabbage,Fruits & Vegetables,12,95,83
Cucumber,Fruits & Vegetables,12,93,81
Egg (Turkey),Dairy,19,100,81
Haddock,Seafood,19,99,80
Evaporated Milk,Dairy,23,100,77
Haddock,Seafood,11,87,76
Evaporated Milk,Dairy,23,99,76
Avocado Oil,Oils & Fats,10,86,76
Basmati Rice,Grains & Pulses,11,87,76


-- VISUAL 5: PIE CHART - Inventory by Category (% of total)

In [ ]:
%sql
WITH category_values AS (
  SELECT
    Category,
    SUM(Stock_Quantity * Unit_Price) AS category_value
  FROM grocery_inventory_adjusted
  GROUP BY Category
),
total_value AS (
  SELECT SUM(Stock_Quantity * Unit_Price) AS total_value
  FROM grocery_inventory_adjusted
)
SELECT
  cv.Category,
  cv.category_value,
  ROUND((cv.category_value * 100.0 / tv.total_value), 2) AS percentage_of_total
FROM category_values cv
CROSS JOIN total_value tv
ORDER BY cv.category_value DESC;

Category,category_value,percentage_of_total
Fruits & Vegetables,86033.16000000003,25.86
Beverages,62942.25,18.92
Seafood,62515.899999999994,18.79
Dairy,50601.95,15.21
Grains & Pulses,31969.2,9.61
Oils & Fats,17211.5,5.17
Bakery,16788.799999999996,5.05
null,4591.95,1.38
